# Mesh Generation in Gmsh: Simple Model

    This notebook creates a simplified 3D reservoir model composed of five geological layers and two wells using Gmsh. The purpose of this model is to provide a clear and reproducible example of the pipeline for geometry construction, mesh generation, and numerical simulation in OpenGeoSys (OGS).

    The notebook is organized into the following main sections:

    1. Geometry definition  
    2. Geometry fragmentation  
    3. Physical group assignment  
    4. Mesh generation  
    5. VTU conversion and export
    6. Visualization & Finalization

In [27]:
# Gmsh Initialization

import gmsh

gmsh.initialize()
gmsh.model.add("mesh6")

# gmsh.option → settings
# gmsh.model → current geometric model you are building (volumes, physical groups, mesh)
# gmsh.model.occ → geometry operations (boxes, cylinders, fragment, com)
# gmsh.fltk → graphical interface tools

   # 1. Geometry definition

    This section defines the dimensions of the model's main components: five geological layers and two wells.
    
<div style="text-align: center;">
    <img src="../images/simple_model.png" width="700">
</div>

<div style="text-align: center;">
    <i>Figure 1. Structured view of the simplified reservoir mesh (visualization in ParaView).</i>
</div>

    The parameters `Lx` and `Ly` define the reservoir dimensions in the x and y directions, while the `z` coordinates define the thickness and position of each geological interface. 

    Layers 2 and 4 represent low-permeability cap rocks that hydraulically isolate the reservoir layer, while Layers 1 and 5 act as outer boundary layers with no hydraulic connection to the simplified flow system considered in this model.

    The two wells are located at opposite corners of the domain (one injector and one producer) and extend vertically to the midpoint of Layer 3 (the reservoir layer).


In [28]:
# length in x,y
Lx = 10.0
Ly = 10.0

# height in z (elevation)

z1 = 5.0        #<---- surface
z2 = 3.5        
z3 = 3.0        
z4 = 2.0        
z5 = 1.5        
z0 = 0.0        #<---- bottom

# Layer 1 (surface): z2 to z1
# Layer 2 (cap rock): z3 to z2
# Layer 3 (reservoir): z4 to z3   <-- layer of interest
# Layer 4 (cap rock): z5 to z4
# Layer 5 (bottom): z0 to z5

# layer 3 = Middle point for well completition
z_middle_central = (z4 + z3) / 2.0   #----------2.5

# Well radius
rw = 0.2

# Well positions (in opposite corners)

xw1, yw1 = 1.0, 1.0
xw2, yw2 = 9.0, 9.0


### Gmsh Primitive Objects: Layers (`addBox`)

    The geological layers are generated as stacked box volumes using the `addBox` function. Each box represents a different geological unit, with its vertical position and thickness controlled by the previously defined `z` coordinates.

In [5]:
# Creating 5 layers = 5 stacked boxes
# gmsh.model.occ.addBox(x, y, z, dx, dy, dz)
# Starting corner point (x, y, z)
# ----------------extend it by (dx, dy, dz)

layer1 = gmsh.model.occ.addBox(0, 0, z2, Lx, Ly, z1 - z2)
layer2 = gmsh.model.occ.addBox(0, 0, z3, Lx, Ly, z2 - z3)
layer3 = gmsh.model.occ.addBox(0, 0, z4, Lx, Ly, z3 - z4)
layer4 = gmsh.model.occ.addBox(0, 0, z5, Lx, Ly, z4 - z5)
layer5 = gmsh.model.occ.addBox(0, 0, z0, Lx, Ly, z5 - z0)

# Layer 1 (upper layer): z1 to z2  <-- surface
# Layer 2 (cap rock): z2 to z3
# Layer 3 (reservoir): z3 to z4   <-- layer of interest
# Layer 4 (cap rock): z4 to z5
# Layer 5 (bottom):  z5 - first layer (z0)

# tag = entity’s ID assigned by Gmsh (dim, tag)
# com = Center of mass

for tag in [layer1, layer2, layer3, layer4, layer5]:
    com = gmsh.model.occ.getCenterOfMass(3, tag)
    print(f"Layer {tag} center of mass = {com}")

Layer 1 center of mass = (5.0, 5.0, 4.25)
Layer 2 center of mass = (5.0, 5.0, 3.25)
Layer 3 center of mass = (5.0, 5.0, 2.5)
Layer 4 center of mass = (5.0, 5.0, 1.7500000000000002)
Layer 5 center of mass = (5.0, 5.0, 0.75)


### Gmsh Primitive Objects: Wells (`addCylinder`)

    The wells are generated as vertical cylindrical volumes using the `addCylinder` function. These cylinders represent the injector and producer wells positioned at opposite corners of the reservoir domain and extending vertically toward the reservoir layer.

In [6]:

well_length = z1 - z_middle_central   # From top surface (z1) down to middle of central layer (z_middle_central)

#addCylinder(x, y, z, dx, dy, dz, r)

well1 = gmsh.model.occ.addCylinder(xw1, yw1, z1, 0, 0, -well_length, rw)
well2 = gmsh.model.occ.addCylinder(xw2, yw2, z1, 0, 0, -well_length, rw)


com1 = gmsh.model.occ.getCenterOfMass(3, well1)
com2 = gmsh.model.occ.getCenterOfMass(3, well2)

print(f"Well 1 → tag = {well1}, center of mass = {com1}")
print(f"Well 2 → tag = {well2}, center of mass = {com2}")
print(well_length, z_middle_central)

Well 1 → tag = 6, center of mass = (1.0, 1.0, 3.75)
Well 2 → tag = 7, center of mass = (8.999999999999998, 8.999999999999998, 3.75)
2.5 2.5


# 2. Geometry Fragmentation

    The `fragment` function in Gmsh splits intersecting volumes and reconnects them into a single consistent geometry. This operation ensures that the wells and geological layers share common interfaces and remain physically connected within the mesh.

    This step is essential for OpenGeoSys (OGS), since it allows the software to correctly identify material regions, boundaries, and hydraulic interactions in later stages (physical group assignment).

In [7]:
#                 (dimension, entity tags)
domain_volumes = [(3, layer1), (3, layer2), (3, layer3), (3, layer4), (3, layer5)]
well_volumes = [(3, well1), (3, well2)]


# fragment() → boolean partitioning operation, splits complex geometries or networks into subsets using boolean logic (AND, OR, NOT) to 
#              simplify, simulate, or analyze subcomponents, such as separating combined 3D shapes (union/subtraction) or decomposing Boolean networks. 
# synchronize()→ updates the main Gmsh model after geometry operations. This allows the newly created or modified volumes, surfaces, 
#                and interfaces to become available for meshing, visualization, and physical group assignment.

gmsh.model.occ.fragment(domain_volumes, well_volumes)

gmsh.model.occ.synchronize()

### Geometry Verification After Fragmentation

    After the fragmentation process, Gmsh automatically generates new volume entities and assigns new tags to each resulting region. This section retrieves the center of mass (COM) of every volume (getEntities(3)) to help identify and verify the location of the fragmented layers and well sections within the model.

In [8]:
for dim, tag in gmsh.model.getEntities(3):
    x, y, z = gmsh.model.occ.getCenterOfMass(dim, tag)
    
    print(f"Volume {tag} | COM = ({x:.2f}, {y:.2f}, {z:.2f})")

Volume 4 | COM = (5.00, 5.00, 1.75)
Volume 5 | COM = (5.00, 5.00, 0.75)
Volume 6 | COM = (5.00, 5.00, 4.25)
Volume 7 | COM = (1.00, 1.00, 4.25)
Volume 8 | COM = (9.00, 9.00, 4.25)
Volume 9 | COM = (5.00, 5.00, 3.25)
Volume 10 | COM = (1.00, 1.00, 3.25)
Volume 11 | COM = (9.00, 9.00, 3.25)
Volume 12 | COM = (5.00, 5.00, 2.50)
Volume 13 | COM = (1.00, 1.00, 2.75)
Volume 14 | COM = (9.00, 9.00, 2.75)


In [9]:
# Surface Verification (getEntities(2)) after Fragmentation
# The coordinates help identify the position of external boundaries and well interfaces, which will later be used for boundary condition assignment and physical grouping in OpenGeoSys (OGS).

for dim, tag in gmsh.model.getEntities(2):
    x, y, z = gmsh.model.occ.getCenterOfMass(dim, tag)
    
    print(f"Surface {tag} | COM = ({x:.2f}, {y:.2f}, {z:.2f})")

Surface 1 | COM = (0.00, 5.00, 4.25)
Surface 2 | COM = (5.00, 0.00, 4.25)
Surface 3 | COM = (5.00, 5.00, 5.00)
Surface 4 | COM = (5.00, 10.00, 4.25)
Surface 5 | COM = (5.00, 5.00, 3.50)
Surface 6 | COM = (10.00, 5.00, 4.25)
Surface 7 | COM = (1.00, 1.00, 4.25)
Surface 8 | COM = (9.00, 9.00, 4.25)
Surface 9 | COM = (1.00, 1.00, 5.00)
Surface 10 | COM = (1.00, 1.00, 3.50)
Surface 11 | COM = (9.00, 9.00, 5.00)
Surface 12 | COM = (9.00, 9.00, 3.50)
Surface 13 | COM = (0.00, 5.00, 3.25)
Surface 14 | COM = (5.00, 0.00, 3.25)
Surface 15 | COM = (5.00, 10.00, 3.25)
Surface 16 | COM = (5.00, 5.00, 3.00)
Surface 17 | COM = (10.00, 5.00, 3.25)
Surface 18 | COM = (1.00, 1.00, 3.25)
Surface 19 | COM = (9.00, 9.00, 3.25)
Surface 20 | COM = (1.00, 1.00, 3.00)
Surface 21 | COM = (9.00, 9.00, 3.00)
Surface 22 | COM = (0.00, 5.00, 2.50)
Surface 23 | COM = (5.00, 0.00, 2.50)
Surface 24 | COM = (5.00, 10.00, 2.50)
Surface 25 | COM = (5.00, 5.00, 2.00)
Surface 26 | COM = (10.00, 5.00, 2.50)
Surface 27 | CO

# 3. Physical Group Assignment

    This section assigns physical groups to the fragmented volumes and surfaces generated in the previous steps. Physical groups act as labels that allow OpenGeoSys (OGS) to identify different regions of the model through physical tags.

    These tags are later used to assign material properties, boundary conditions, source terms, and other physical parameters required during the simulation setup.

In [10]:
# -----------------------------
# GEOLOGICAL LAYERS / VOLUMES (3)
# gmsh.model.addPhysicalGroup(dim, tags, physical_tag) --> physical_tag: numerical identifier used by OpenGeoSys (OGS) to recognize the region.
# gmsh.model.setPhysicalName(dim, physical_tag, "name") --> name: assigns a descriptive name to the physical group.
# 
# ******NOTE******
# The entity tags used in the physical groups were identified from the Center of Mass (COM) coordinates obtained after fragmentation.
# Reference table:
# simple_model.xlsx
# -----------------------------

gmsh.model.addPhysicalGroup(3, [6], 105)
gmsh.model.setPhysicalName(3, 105, "Layer1_Top")

gmsh.model.addPhysicalGroup(3, [9], 104)
gmsh.model.setPhysicalName(3, 104, "Layer2_caprock1")

gmsh.model.addPhysicalGroup(3, [12], 103)
gmsh.model.setPhysicalName(3, 103, "Layer3_Reservoir")

gmsh.model.addPhysicalGroup(3, [4], 102)
gmsh.model.setPhysicalName(3, 102, "Layer4_caprock2")

gmsh.model.addPhysicalGroup(3, [5], 101)
gmsh.model.setPhysicalName(3, 101, "Layer5_bottom")


# -----------------------------
# WELL 1 / VOLUMES (3)
# -----------------------------
gmsh.model.addPhysicalGroup(3, [13, 10, 7], 201)
gmsh.model.setPhysicalName(3, 201, "Well1")

gmsh.model.addPhysicalGroup(3, [7], 202)
gmsh.model.setPhysicalName(3, 202, "Well1_upper")

gmsh.model.addPhysicalGroup(3, [10], 203)
gmsh.model.setPhysicalName(3, 203, "Well1_middle")

gmsh.model.addPhysicalGroup(3, [13], 204)
gmsh.model.setPhysicalName(3, 204, "Well1_bottom")

# -----------------------------
# WELL 2 / VOLUMES (3)
# -----------------------------
gmsh.model.addPhysicalGroup(3, [14, 11, 8], 205)
gmsh.model.setPhysicalName(3, 205, "Well2")

gmsh.model.addPhysicalGroup(3, [8], 206)
gmsh.model.setPhysicalName(3, 206, "Well2_upper")

gmsh.model.addPhysicalGroup(3, [11], 207)
gmsh.model.setPhysicalName(3, 207, "Well2_middle")

gmsh.model.addPhysicalGroup(3, [14], 208)
gmsh.model.setPhysicalName(3, 208, "Well2_bottom")


for dim, tag in gmsh.model.getPhysicalGroups():
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

3 101 Layer5_bottom
3 102 Layer4_caprock2
3 103 Layer3_Reservoir
3 104 Layer2_caprock1
3 105 Layer1_Top
3 201 Well1
3 202 Well1_upper
3 203 Well1_middle
3 204 Well1_bottom
3 205 Well2
3 206 Well2_upper
3 207 Well2_middle
3 208 Well2_bottom


In [11]:
# --------------------------------------------------
# WELL 1 / COM of SURFACES (2) based on VOLUMES (3)
# --------------------------------------------------

for v in [7, 10, 13]:
    print("Well1 Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "com =", com)

Well1 Volume 7
  Surface 7 com = (1.0, 1.0, 4.249999999999999)
  Surface 9 com = (1.0, 1.0, 5.0)
  Surface 10 com = (1.0, 1.0, 3.5)
Well1 Volume 10
  Surface 10 com = (1.0, 1.0, 3.5)
  Surface 18 com = (1.0, 1.0, 3.25)
  Surface 20 com = (1.0, 1.0, 3.0)
Well1 Volume 13
  Surface 20 com = (1.0, 1.0, 3.0)
  Surface 27 com = (1.0, 1.0, 2.75)
  Surface 29 com = (1.0, 1.0, 2.5)


In [12]:
# --------------------------------------------------
# WELL 2 / COM of SURFACES (2) based on VOLUMES (3)
# --------------------------------------------------

for v in [8, 11, 14]:
    print("Well2 Volume", v)
    for dim, tag in gmsh.model.getBoundary([(3, v)], oriented=False, recursive=False):
        if dim == 2:
            com = gmsh.model.occ.getCenterOfMass(2, tag)
            print("  Surface", tag, "com =", com)

Well2 Volume 8
  Surface 8 com = (9.0, 9.0, 4.249999999999999)
  Surface 11 com = (9.0, 9.0, 5.0)
  Surface 12 com = (9.0, 9.0, 3.5)
Well2 Volume 11
  Surface 12 com = (9.0, 9.0, 3.5)
  Surface 19 com = (9.0, 9.0, 3.25)
  Surface 21 com = (9.0, 9.0, 3.0)
Well2 Volume 14
  Surface 21 com = (9.0, 9.0, 3.0)
  Surface 28 com = (9.0, 9.0, 2.75)
  Surface 30 com = (9.0, 9.0, 2.5)


In [13]:
# --------------------------------------------------
# Create physical groups for well surfaces:
#
# ******NOTE******
# The entity tags used in the physical groups were identified from the Center of Mass (COM) coordinates obtained after fragmentation.
# Reference table:
# simple_model.xlsx
# --------------------------------------------------

# -----------------------------
# WELL 1 SURFACES (2)
# -----------------------------
gmsh.model.addPhysicalGroup(2, [9], 401)
gmsh.model.setPhysicalName(2, 401, "Well1_Top")

gmsh.model.addPhysicalGroup(2, [27], 402)
gmsh.model.setPhysicalName(2, 402, "Well1_compl")

# -----------------------------
# WELL 2 SURFACES (2)
# -----------------------------
gmsh.model.addPhysicalGroup(2, [11], 403)
gmsh.model.setPhysicalName(2, 403, "Well2_Top")

gmsh.model.addPhysicalGroup(2, [28], 404)
gmsh.model.setPhysicalName(2, 404, "Well2_compl")

for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))


2 401 Well1_Top
2 402 Well1_compl
2 403 Well2_Top
2 404 Well2_compl


In [14]:
# --------------------------------------------------
# Create physical groups for the formation 
#    (outer boundaries of the box)
#
# ******NOTE******
# The entity tags used in the physical groups were identified from the Center of Mass (COM) coordinates obtained after fragmentation.
# Reference table:
# simple_model.xlsx
# --------------------------------------------------

# Left boundary = Xmin (x = 0, y = 5, z = com z)
gmsh.model.addPhysicalGroup(2, [1, 13, 22, 31, 36], 301)
gmsh.model.setPhysicalName(2, 301, "Left")

# Right boundary = Xmax (x = 10, y = 5, z = com z)
gmsh.model.addPhysicalGroup(2, [6, 17, 26, 32, 37], 302)
gmsh.model.setPhysicalName(2, 302, "Right")

# Front boundary = Ymin (x = 5, y = 0, z = com z)
gmsh.model.addPhysicalGroup(2, [2, 14, 23, 33, 38], 303)
gmsh.model.setPhysicalName(2, 303, "Front")

# Back boundary = Ymax (x = 5, y = 10, z = com z)
gmsh.model.addPhysicalGroup(2, [4, 15, 24, 34, 39], 304)
gmsh.model.setPhysicalName(2, 304, "Back")

# Top boundary = Ztop (x = 5, y = 5, z = 5)
gmsh.model.addPhysicalGroup(2, [3], 305)
gmsh.model.setPhysicalName(2, 305, "Top")

# Bottom boundary = Zmin (x = 5, y = 5, z = 0)
gmsh.model.addPhysicalGroup(2, [40], 306)
gmsh.model.setPhysicalName(2, 306, "Bottom")

for dim, tag in gmsh.model.getPhysicalGroups(2):
    print(dim, tag, gmsh.model.getPhysicalName(dim, tag))

2 301 Left
2 302 Right
2 303 Front
2 304 Back
2 305 Top
2 306 Bottom
2 401 Well1_Top
2 402 Well1_compl
2 403 Well2_Top
2 404 Well2_compl


# 4. Mesh Generation

    Global and local mesh sizes are assigned to control the resolution of the numerical model, with finer refinement applied near regions of interest (wells and the reservoir [layer 3]).

In [15]:
# --------------------------------------------------
# GLOBAL MESH SIZES
#    Assign a general size to all points
#    In Gmsh, the mesh size is assigned to points, surfaces and 
#    volumes inherit mesh size from their corner points unless overridden.
# --------------------------------------------------

#            gmsh.model.getEntities(dim) --> retrieves all geometric entities of a specified dimension.

all_points = gmsh.model.getEntities(0) # dim = 0 --> collects all point entities in the model.

# gmsh.model.mesh.setSize(entity_list, size) 
#                                      size ----> A global mesh size at the listed points

gmsh.model.mesh.setSize(all_points, 2)


In [16]:
# --------------------------------------------------------
# REFINEMENT BY LAYER USING BOX FIELDS
#
# gmsh.model.mesh.field.add("Box")
# ---> Creates a Box field, defining a rectangular 3D
#      region where a different mesh size can be assigned.
#
# gmsh.model.mesh.field.setNumber(field, parameter, value)
# ---> Assigns numerical values to the selected mesh field.
#
# Parameters:
# VIn,  VOut  ---> mesh size inside and outside the box
# XMin, XMax  ---> minimum and maximum x-coordinate
# YMin, YMax  ---> minimum and maximum y-coordinate
# ZMin, ZMax  ---> minimum and maximum z-coordinate (vertical)
#
# --------------------------------------------------------

# --------------------------------------------------------
# Layer 1 and 5: coarsest
# --------------------------------------------------------

field1 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field1, "VIn", 2)
gmsh.model.mesh.field.setNumber(field1, "VOut", 2)
gmsh.model.mesh.field.setNumber(field1, "XMin", 0)
gmsh.model.mesh.field.setNumber(field1, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field1, "YMin", 0)
gmsh.model.mesh.field.setNumber(field1, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field1, "ZMin", z2)
gmsh.model.mesh.field.setNumber(field1, "ZMax", z1)

field5 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field5, "VIn", 2)
gmsh.model.mesh.field.setNumber(field5, "VOut", 2)
gmsh.model.mesh.field.setNumber(field5, "XMin", 0)
gmsh.model.mesh.field.setNumber(field5, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field5, "YMin", 0)
gmsh.model.mesh.field.setNumber(field5, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field5, "ZMin", z0)
gmsh.model.mesh.field.setNumber(field5, "ZMax", z5)


In [17]:
# --------------------------------------------------------
# Layer 3: most refined
# --------------------------------------------------------

field3 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field3, "VIn", 0.2)
gmsh.model.mesh.field.setNumber(field3, "VOut", 1)
gmsh.model.mesh.field.setNumber(field3, "XMin", 0)
gmsh.model.mesh.field.setNumber(field3, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field3, "YMin", 0)
gmsh.model.mesh.field.setNumber(field3, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field3, "ZMin", z4)
gmsh.model.mesh.field.setNumber(field3, "ZMax", z3)


In [18]:
# --------------------------------------------------------
# Layer 2 and 4 - cap rocks: intermediate
# --------------------------------------------------------

field2 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field2, "VIn", 1)
gmsh.model.mesh.field.setNumber(field2, "VOut", 1)
gmsh.model.mesh.field.setNumber(field2, "XMin", 0)
gmsh.model.mesh.field.setNumber(field2, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field2, "YMin", 0)
gmsh.model.mesh.field.setNumber(field2, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field2, "ZMin", z3)
gmsh.model.mesh.field.setNumber(field2, "ZMax", z2)

field4 = gmsh.model.mesh.field.add("Box")
gmsh.model.mesh.field.setNumber(field4, "VIn", 1)
gmsh.model.mesh.field.setNumber(field4, "VOut", 1)
gmsh.model.mesh.field.setNumber(field4, "XMin", 0)
gmsh.model.mesh.field.setNumber(field4, "XMax", Lx)
gmsh.model.mesh.field.setNumber(field4, "YMin", 0)
gmsh.model.mesh.field.setNumber(field4, "YMax", Ly)
gmsh.model.mesh.field.setNumber(field4, "ZMin", z5)
gmsh.model.mesh.field.setNumber(field4, "ZMax", z4)


In [19]:
# --------------------------------------------------
# EXTRA REFINEMENT AROUND WELLS
#    Cylinder fields define a cylindrical region in 3D space
#    XCenter, YCenter, ZCenter → the center of the cylinder
#    XAxis, YAxis, ZAxis → the direction vector of the cylinder’s axis
#    Radius → cylinder radius
# --------------------------------------------------

well_field1 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field1, "VIn", 0.3)
gmsh.model.mesh.field.setNumber(well_field1, "VOut", 0.3)
gmsh.model.mesh.field.setNumber(well_field1, "XCenter", xw1)
gmsh.model.mesh.field.setNumber(well_field1, "YCenter", yw1)
gmsh.model.mesh.field.setNumber(well_field1, "ZCenter", 3.75) # COM Well1
gmsh.model.mesh.field.setNumber(well_field1, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field1, "ZAxis", 1) # unit vector along Z
gmsh.model.mesh.field.setNumber(well_field1, "Radius", rw)

well_field2 = gmsh.model.mesh.field.add("Cylinder")
gmsh.model.mesh.field.setNumber(well_field2, "VIn", 0.3)
gmsh.model.mesh.field.setNumber(well_field2, "VOut", 0.3)
gmsh.model.mesh.field.setNumber(well_field2, "XCenter", xw2)
gmsh.model.mesh.field.setNumber(well_field2, "YCenter", yw2)
gmsh.model.mesh.field.setNumber(well_field2, "ZCenter", 3.75) # COM Well2
gmsh.model.mesh.field.setNumber(well_field2, "XAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "YAxis", 0)
gmsh.model.mesh.field.setNumber(well_field2, "ZAxis", 1) # unit vector along Z = vertical cylinder
gmsh.model.mesh.field.setNumber(well_field2, "Radius", rw)


### Mesh Refinement Combination and Background Mesh

    This section combines all previously defined local refinement fields into a single mesh refinement strategy. The different refinement regions assigned to geological layers and wells are merged using a `Min` field, which automatically selects the smallest mesh size from all active refinement fields at each location in the model.

    The combined field is then used by Gmsh as the main reference for mesh generation, determining the final mesh size across the entire geometry.

In [20]:
# --------------------------------------------------
# COMBINE FIELDS
#    Merge all your local refinement fields (layer boxes + well cylinders) into a single background field.
#
#    gmsh.model.mesh.field.add("Min") → creates a Min field = selects the smallest mesh size at each location of the geometry.
#    gmsh.model.mesh.field.setNumbers(field, parameter, values)
#                                                       values → [field1, field2, field3] → list of mesh fields to combine
#    gmsh.model.mesh.field.setAsBackgroundMesh(field) → assigns the combined refinement field as the main mesh size controller
#    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0) → disable automatic mesh refinement methods in Gmsh
#    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0) → disable automatic mesh refinement methods in Gmsh
# --------------------------------------------------
min_field = gmsh.model.mesh.field.add("Min") # computes the minimum value of several mesh fields at each point.

gmsh.model.mesh.field.setNumbers(
    min_field,
    "FieldsList",                  #"FieldsList" → predefined Gmsh parameter used by the `Min` field
    [field1, field2, field3, field4, field5, well_field1, well_field2]
)
gmsh.model.mesh.field.setAsBackgroundMesh(min_field) #Min field as the “background mesh”

# Optionals: No point-based or curvature-based automatic sizing
#         Mesh.MeshSizeExtendFromBoundary = 0, avoid automatically extend boundary-based sizes into the interior
#         Mesh.MeshSizeFromPoints = 0, Gmsh ignores point-based sizing as a primary source.
#        Mesh.MeshSizeFromCurvature = 0, not want automatic curvature-based refinement interfering.

gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)


### 3D Mesh Generation

    Final command for generating the three-dimensional computational mesh from the previously defined geometry and refinement fields. 
    
    Gmsh discretizes the geological layers and wells into volumetric mesh elements that will later be exported and used by OpenGeoSys (OGS) for numerical simulations.

In [21]:
# --------------------------------------------------
# GENERATE 3D MESH
# dim = 3 → volumetric 3D mesh
# --------------------------------------------------
gmsh.model.mesh.generate(3)


In [22]:
# --------------------------------------------------
# Mesh Element Verification
# Retrieving the number of mesh elements generated inside each 3D volume after mesh generation. 
# Helps verifying that all geological layers and well regions were successfully meshed and provides a quick overview of the mesh distribution across the model.
#
# gmsh.model.mesh.getElements(dim, tag) → retrieves the mesh elements associated with a specific geometric entity (3D volume).
# --------------------------------------------------

for v in gmsh.model.getEntities(3):
    elem_types, elem_tags, elem_node_tags = gmsh.model.mesh.getElements(3, v[1])
    n_elems = sum(len(tags) for tags in elem_tags)
    print(f"Volume {v[1]} has {n_elems} elements")

Volume 4 has 19240 elements
Volume 5 has 27141 elements
Volume 6 has 27214 elements
Volume 7 has 81 elements
Volume 8 has 75 elements
Volume 9 has 18912 elements
Volume 10 has 47 elements
Volume 11 has 44 elements
Volume 12 has 60386 elements
Volume 13 has 66 elements
Volume 14 has 68 elements


### Mesh Export and Save

    This section defines the mesh export settings and saves the generated mesh into `.msh` format. 

In [23]:
# --------------------------------------------------
# MESH EXPORT
# gmsh.option.setNumber(parameter, value)
# gmsh.write("file_name.msh") → generated mesh to the specified .msh file
#
# Parameter:
#          ("Mesh.MshFileVersion", 2.2) → format version to 2.2 for compatibility
#          ("Mesh.Binary", 0) → mesh in ASCII format (0) instead of binary format (1) = readable and easier to inspect.
# --------------------------------------------------

gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.option.setNumber("Mesh.Binary", 0)


gmsh.write("mesh6.msh")


### Checking physical tags in the mesh

In [24]:
import meshio
import numpy as np

mesh = meshio.read("mesh6.msh")

for i, cell_block in enumerate(mesh.cells):
    vals = mesh.cell_data["gmsh:physical"][i]
    print(f"Block {i}: type = {cell_block.type}, tags = {np.unique(vals)}")


Block 0: type = triangle, tags = [301 302 303 304 305 306 401 402 403 404]
Block 1: type = tetra, tags = [101 102 103 104 105 201 202 203 204 205 206 207 208]


# 5. VTU Conversion and Export

    This section converts the generated Gmsh mesh (`.msh`) into `.vtu` format using `meshio`. The `.vtu` format is required for OpenGeoSys (OGS) simulations and preserves the mesh geometry, element connectivity, and physical group information generated in Gmsh.

<div style="text-align: center;">
    <img src="../images/simple_model_mesh.png" width="700">
</div>

<div style="text-align: center;">
<i>Figure 2. Final three-dimensional computational mesh, showing the discretized geological layers after local mesh refinement (visualization in ParaView).</i>
</div>

    During the conversion process, only the relevant mesh element types (`tetra` and `triangle`) are exported together with their corresponding physical and geometrical tags.

In [25]:
import meshio
import numpy as np
# -----------------------------------------------------------
# meshio creates a Mesh object that stores all mesh information.
#    mesh.points → node coordinates
#    mesh.cells → mesh element connectivity
#    mesh.cell_data → additional data associated with mesh elements (physical & geometrical tags)
# -----------------------------------------------------------

# mesh --> meshio.Mesh object containing the complete mesh information
mesh = meshio.read("mesh6.msh")

cells_out = []
phys_out = []
geom_out = []

for i, cell_block in enumerate(mesh.cells):
    cell_type = cell_block.type
    
    if cell_type in ["tetra", "triangle"]:  # store the physical group tags and geometrical entity tags generated in Gmsh.
        cells_out.append((cell_type, cell_block.data))  
        phys_out.append(mesh.cell_data["gmsh:physical"][i])
        geom_out.append(mesh.cell_data["gmsh:geometrical"][i])

# New mesh object containing the filtered mesh data.

new_mesh = meshio.Mesh(
    points=mesh.points,
    cells=cells_out,
    cell_data={
        "gmsh:physical": phys_out,
        "gmsh:geometrical": geom_out,
    },
)

# exports the mesh into .vtu format
meshio.write("mesh6.vtu", new_mesh)

# 9. Visualization & Finalization

    Opening the Gmsh graphical interface for direct visualization of the generated geometry and mesh. After visualization, the Gmsh session is properly closed to release memory and finalize the workflow execution.

In [26]:

gmsh.fltk.run()  # opens the Gmsh graphical user interface (GUI)

gmsh.finalize()  # closes the Gmsh session